# AI Drug Discovery Simulator — Batch Analysis Demo
Screen multiple drug candidates in one call using the batch API endpoint.
No UI required — pure Python research workflow.

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

API_BASE = 'http://localhost:8000'

In [ ]:
# Define your candidate molecules
molecules = [
    'imatinib',
    'warfarin',
    'aspirin',
    'ibuprofen',
    'caffeine',
    'tamoxifen',
    'metformin',
    'amiodarone',
    'paracetamol',
]

# Run batch analysis
response = requests.post(
    f'{API_BASE}/api/v1/batch/analyze',
    json={'molecules': molecules}
)
data = response.json()
print(f"Analyzed {data['successful']}/{data['total']} molecules in {data['processing_time_ms']:.0f}ms")

In [ ]:
# Load into DataFrame — filter successful only
df = pd.DataFrame([r for r in data['results'] if r['status'] == 'success'])

# Display ranked table
display_cols = ['rank', 'molecule_name', 'grade', 'druggability_score', 'pkd', 'admet_score', 'molecular_weight', 'lipinski_pass']
df[display_cols].set_index('rank')

In [ ]:
# Druggability score bar chart
grade_colors = {'A': '#22c55e', 'B': '#eab308', 'C': '#f97316', 'D': '#ef4444'}
colors = [grade_colors.get(g, '#6b7280') for g in df['grade']]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d0d0f')

# --- Plot 1: Druggability Score ---
ax1 = axes[0]
ax1.set_facecolor('#111113')
bars = ax1.barh(df['molecule_name'], df['druggability_score'], color=colors, height=0.6)
ax1.set_xlabel('Druggability Score (0-100)', color='#9ca3af', fontsize=10)
ax1.set_title('Druggability Ranking', color='white', fontsize=12, fontweight='bold', pad=12)
ax1.tick_params(colors='#9ca3af')
ax1.spines[:].set_color('#27272a')
ax1.set_xlim(0, 100)
for bar, score in zip(bars, df['druggability_score']):
    ax1.text(score + 1, bar.get_y() + bar.get_height()/2,
             f'{score:.1f}', va='center', color='white', fontsize=9)

legend = [mpatches.Patch(color=c, label=f'Grade {g}') for g, c in grade_colors.items()]
ax1.legend(handles=legend, loc='lower right', facecolor='#1c1c1e', labelcolor='white', fontsize=8)

# --- Plot 2: pKd vs ADMET scatter ---
ax2 = axes[1]
ax2.set_facecolor('#111113')
scatter = ax2.scatter(df['pkd'], df['admet_score'], c=colors, s=120, zorder=3, edgecolors='white', linewidths=0.5)
for _, row in df.iterrows():
    ax2.annotate(row['molecule_name'], (row['pkd'], row['admet_score']),
                 textcoords='offset points', xytext=(6, 4),
                 color='#9ca3af', fontsize=8)
ax2.set_xlabel('Binding Affinity (pKd)', color='#9ca3af', fontsize=10)
ax2.set_ylabel('ADMET Score', color='#9ca3af', fontsize=10)
ax2.set_title('Binding vs Safety Profile', color='white', fontsize=12, fontweight='bold', pad=12)
ax2.tick_params(colors='#9ca3af')
ax2.spines[:].set_color('#27272a')
ax2.grid(True, color='#27272a', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('batch_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0d0d0f')
plt.show()
print('Chart saved to batch_analysis.png')

In [ ]:
# Print top 3 candidates with recommendation
print('=== TOP CANDIDATES ===')
for _, row in df.head(3).iterrows():
    print(f"\n#{int(row['rank'])} {row['molecule_name'].upper()}")
    print(f"   Grade: {row['grade']}  |  Druggability: {row['druggability_score']:.1f}/100")
    print(f"   pKd: {row['pkd']:.2f}  |  ADMET: {row['admet_score']:.1f}/100")
    print(f"   Lipinski: {'PASS' if row['lipinski_pass'] else 'FAIL'}  |  MW: {row['molecular_weight']:.1f} Da")